In [30]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.track(
    source="MOT17-02-FRCNN/img1",
    tracker="botsort.yaml",
    conf=0.3,
    iou=0.5,
    classes=[0], #person
    save=True,
    persist=True
)


image 1/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000001.jpg: 384x640 9 persons, 70.0ms
image 2/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000002.jpg: 384x640 8 persons, 77.0ms
image 3/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000003.jpg: 384x640 8 persons, 77.8ms
image 4/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000004.jpg: 384x640 9 persons, 75.6ms
image 5/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000005.jpg: 384x640 9 persons, 78.6ms
image 6/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000006.jpg: 384x640 9 persons, 71.8ms
image 7/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000007.jpg: 384x640 9 persons, 72.1ms
image 8/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000008.jpg: 384x640 9 persons, 68.7ms
image 9/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\000009.jpg: 384x640 10 persons, 76.0ms
image 10/600 d:\record_by_me\IPr\CV\FinalTerm\MOT17-02-FRCNN\img1\00001

In [31]:
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from scipy.optimize import linear_sum_assignment

In [ ]:
# 1. Save tracking results to video
def save_video(destination):    
    frames = sorted(Path("runs/detect/track").glob("*.jpg"))
    frame = cv2.imread(str(frames[0]))
    h, w = frame.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(destination, fourcc, 30, (w, h))

    for f in frames:
        img = cv2.imread(str(f))
        out.write(img)
    out.release()
    print(f"Video saved: {len(frames)} frames")
# 2. Load GT and predictions
def load_gt():
    gt_data = []
    with open("MOT17-02-FRCNN/gt/gt.txt") as f:
        for line in f:
            parts = [float(x) for x in line.strip().split(",")]
            # MOT17 format: frame, track_id, x, y, w, h, mark, class_id, visibility
            frame = int(parts[0])
            track_id = int(parts[1])
            x = parts[2]
            y = parts[3]
            w = parts[4]
            h = parts[5]
            mark = int(parts[6])  # 1=evaluatable, 0=ignore
            class_id = int(parts[7])  # 1=Pedestrian, 7=Static, 8=Distractor, 9=Occluded
            visibility = parts[8]  # 0.0-1.0
            
            if mark == 1 and (class_id == 1 or class_id == 7) and visibility > 0.3:
                gt_data.append({
                    "frame": frame, 
                    "gt_id": track_id, 
                    "x": x, 
                    "y": y, 
                    "w": w, 
                    "h": h
                })
    return pd.DataFrame(gt_data)
def load_predictions(results):
    pred_data = []
    for idx, r in enumerate(results):
        frame_num = idx + 1
        if r.boxes.id is not None:
            for box, tid in zip(r.boxes.xyxy, r.boxes.id):
                x1, y1, x2, y2 = box[:4]
                pred_data.append({"frame": frame_num, "pred_id": int(tid), "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2)})
    return pd.DataFrame(pred_data)
# 3. Calculate IoU
def calc_iou(gt, pred):
    x1_1, y1_1, w1, h1 = gt["x"], gt["y"], gt["w"], gt["h"]
    x2_1, y2_1 = x1_1 + w1, y1_1 + h1
    x1_2, y1_2, x2_2, y2_2 = pred["x1"], pred["y1"], pred["x2"], pred["y2"]
    
    xi1, yi1 = max(x1_1, x1_2), max(y1_1, y1_2)
    xi2, yi2 = min(x2_1, x2_2), min(y2_1, y2_2)
    
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1*h1 + (x2_2-x1_2)*(y2_2-y1_2) - inter
    return inter / union if union > 0 else 0
# 4. Match and calculate metrics
def match_frame(gt, pred):
    """Match GT and predictions for one frame"""
    if len(gt) == 0 or len(pred) == 0:
        return 0, len(gt), len(pred), {}
    
    # Compute IoU matrix
    iou_mat = np.zeros((len(gt), len(pred)))
    for i, (_, g) in enumerate(gt.iterrows()):
        for j, (_, p) in enumerate(pred.iterrows()):
            iou_mat[i, j] = calc_iou(g, p)
    
    # Hungarian matching
    gi, pi = linear_sum_assignment(-iou_mat)
    
    # Count TP and track matches
    tp = 0
    matches = {}  # gt_id -> pred_id
    for g_idx, p_idx in zip(gi, pi):
        if iou_mat[g_idx, p_idx] >= 0.5:
            tp += 1
            gt_id = gt.iloc[g_idx]["gt_id"]
            pred_id = pred.iloc[p_idx]["pred_id"]
            matches[gt_id] = pred_id
    
    fn = len(gt) - len(gi)  # Unmatched GT
    fp = len(pred) - len(pi)  # Unmatched Pred
    
    return tp, fn, fp, matches
# 4b. Count metrics across all frames
def count_metrics(gt_df, pred_df):
    """Count TP, FP, FN, ID switches across all frames"""
    tp = fp = fn = id_sw = 0
    prev_match = {}
    
    for frame in sorted(gt_df["frame"].unique()):
        gt = gt_df[gt_df["frame"] == frame]
        pred = pred_df[pred_df["frame"] == frame] if frame in pred_df["frame"].values else pd.DataFrame()
        
        frame_tp, frame_fn, frame_fp, matches = match_frame(gt, pred)
        
        tp += frame_tp
        fn += frame_fn
        fp += frame_fp
        
        # Count ID switches
        for gt_id, pred_id in matches.items():
            if gt_id in prev_match and prev_match[gt_id] != pred_id:
                id_sw += 1
        
        prev_match = matches
    
    return tp, fp, fn, id_sw
# 4c. Calculate final metrics from counts
def calculate_metrics(tp, fp, fn, id_sw, total_gt):
    """
    - MOTA: 1 - (FN + FP + IDS) / total_gt
    - IDF1: 2*P*R / (P + R) where P = TP/(TP+FP), R = TP/(TP+FN)
    - HOTA: sqrt(DetA * AssA) where DetA = TP/(TP+FP+FN), AssA = TP/(TP+IDS)
    """
    mota = 1 - (fn + fp + id_sw) / total_gt if total_gt > 0 else 0
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0  # precision
    r = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall
    idf1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    
    det_a = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0  # detection accuracy
    ass_a = tp / (tp + id_sw) if (tp + id_sw) > 0 else 0  # association accuracy
    hota = np.sqrt(det_a * ass_a)
    
    return mota, idf1, hota

# 4d. Main evaluation function
def evaluate_tracking(gt_df, pred_df):
    tp, fp, fn, id_sw = count_metrics(gt_df, pred_df)
    total_gt = len(gt_df)
    mota, idf1, hota = calculate_metrics(tp, fp, fn, id_sw, total_gt)
    return mota, idf1, hota


In [33]:
save_video("output.mp4")

gt_df = load_gt()
pred_df = load_predictions(results)
mota, idf1, hota = evaluate_tracking(gt_df, pred_df)

print("\n" + "="*40)
print("TRACKING EVALUATION RESULTS")
print("="*40)
print(f"HOTA: {hota*100:.2f}%")
print(f"MOTA: {mota*100:.2f}%")
print(f"IDF1: {idf1*100:.2f}%")

Video saved: 600 frames

TRACKING EVALUATION RESULTS
HOTA: 77.00%
MOTA: 64.07%
IDF1: 74.48%
